# 03 · Join Sofascore + Capology — France Ligue 1 24/25

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2024/25 de Ligue 1 francesa**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_france_2425.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_france_2425.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  542 jugadores | 116 columnas
Capology:   561 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   as monaco
   olympique de marseille
   olympique lyonnais
   paris saint germain
   rc lens
   rc strasbourg
   saint etienne
   stade brestois
   stade de reims
   stade rennais
   usl dunkerque

En Capology pero no en Sofascore:
   brest
   lens
   lyon
   marseille
   monaco
   psg
   reims
   rennes
   st etienne
   strasbourg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brest':'stade brestois',
            'lens':'rc lens',
            'lyon':'olympique lyonnais',
            'marseille':'olympique de marseille',
            'monaco':'as monaco',
            'psg':'paris saint germain',
            'reims':'stade de reims',
            'rennes':'stade rennais',
            'st etienne':'saint etienne',
            'strasbourg':'rc strasbourg'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 445/542 (82.1%)
Sin emparejar: 97


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          7
Revisión media    (0.75 ≤ score < 0.90):   12
Revisión estricta (0.50 ≤ score < 0.75):   39
Revisión muy est. (score < 0.50):           38


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
4,Przemysław Frankowski,RC Lens,przemyslaw frankowski,0.976
18,Radosław Majecki,AS Monaco,radoslaw majecki,0.968
13,Emmanuel Emegha,RC Strasbourg,emanuel emegha,0.966
5,Marcin Bułka,Nice,marcin bulka,0.957
23,Luc Zogbé,Stade Brestois,luck zogbe,0.947
25,Albert Grønbæk,Stade Rennais,albert grnbaek,0.923
30,Bahereba Guirassy,Nantes,herba guirassy,0.903


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
19,Mousa Tamari,Stade Rennais,mousa al tamari,0.889
16,Valentin Atangana Edoa,Stade de Reims,valentin atangana,0.872
3,Đorđe Petrović,RC Strasbourg,djordje petrovic,0.857
70,Justin-Noel Kalumba,Angers,justin kalumba,0.848
17,Matías Fernández,Lille,matias fernandez pardo,0.842
39,Lucas Mincarelli Davin,Montpellier,lucas mincarelli,0.842
15,Henrik Wendel Meister,Stade Rennais,henrik meister,0.800
31,Shavy Warren Babicka,Toulouse,shavy babicka,0.788
29,Thelonius Bair,Auxerre,theo bair,0.783
40,Andrés Gómez,Stade Rennais,carlos andres gomez,0.774


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 12 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
12,Amir Murillo,Olympique de Marseille,michael murillo,0.741
50,Mohamed Bamba,Stade de Reims,mohamed daramy,0.741
46,Benjamin Old,Saint-Étienne,ben old,0.737
45,Ahmadou Bamba Dieng,Angers,bamba dieng,0.733
7,Alexsandro Ribeiro,Lille,alexsandro,0.714
10,Gift Orban,Olympique Lyonnais,gift emmanuel orban,0.690
9,Mutassim Al-Musrati,AS Monaco,al musrati,0.690
8,Abdulay Juma Bah,RC Lens,juma bah,0.667
71,David Pereira da Costa,RC Lens,david costa,0.667
77,Kandet Diawara,Le Havre,mahamadou diawara,0.645


In [14]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['amir murillo',
                    'benjamin old',
                    'ahmadou bamba dieng',
                    'alexsandro ribeiro',
                    'gift orban',
                    'mutassim al musrati',
                    'abdulay juma bah',
                    'david pereira da costa',
                    'boubakar kouyate',
                    'andy logbo',
                    'abner vinicius'
                    
                    
                    
                    

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 11


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [15]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
89,Jordan Lotomba,Nice,jonathan clauss,0.483
88,Saimon Bouabre,AS Monaco,folarin balogun,0.483
74,Hamidou Makalou,Stade Brestois,kamory doumbia,0.483
67,Prosper Peter,Angers,pierrick capelle,0.483
35,Plamedi Nsingi,Nantes,moses simon,0.480
54,Lucas Michal,AS Monaco,lamine camara,0.480
55,Junior Ndiaye,Montpellier,jordan ferri,0.480
34,Robinio Vaz,Olympique de Marseille,geronimo rulli,0.480
78,Alexi Koum,Olympique de Marseille,azzedine ounahi,0.480
75,Simon Cara,Montpellier,lucas mincarelli,0.462


In [16]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [17]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 475/542 (87.6%)
Sin salario:     67


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [18]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 67


,player,team,minutesPlayed,appearances,goals,assists
0,Ismail Jakobs,AS Monaco,180,2,0,0
1,Lucas Michal,AS Monaco,114,8,0,0
2,Saimon Bouabre,AS Monaco,96,3,0,0
3,Mamadou Coulibaly,AS Monaco,13,1,0,0
4,Enzo Caumont,Angers,37,2,0,0
5,Prosper Peter,Angers,27,2,0,0
6,Lanroy Machine,Angers,9,2,0,0
7,Rudy Matondo,Auxerre,62,5,0,0
8,Yoann Cissé,Auxerre,14,2,0,0
9,Mathéo Bodmer,Le Havre,14,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [19]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  AS Monaco  —  SF sin salario:


,player,minutesPlayed
0,Ismail Jakobs,180
1,Lucas Michal,114
2,Mamadou Coulibaly,13
3,Saimon Bouabre,96


  CG plantilla completa:


,player,player_norm
0,Al Musrati,al musrati
1,Aleksandr Golovin,aleksandr golovin
2,Breel Embolo,breel embolo
3,Caio Henrique,caio henrique
4,Christian Mawissa,christian mawissa
5,Denis Zakaria,denis zakaria
6,Edan Diop,edan diop
7,Eliesse Ben Seghir,eliesse ben seghir
8,Eliot Matazo,eliot matazo
9,Folarin Balogun,folarin balogun



  Angers  —  SF sin salario:


...

,player,player_norm
0,Álex Domínguez,alex dominguez
1,Aron Dønnum,aron dnnum
2,Charlie Cresswell,charlie cresswell
3,Cristian Cásseres Jr,cristian casseres jr
4,Denis Genreau,denis genreau
5,Djibril Sidibé,djibril sidibe
6,Frank Magri,frank magri
7,Gabriel Suazo,gabriel suazo
8,Guillaume Restes,guillaume restes
9,Joshua King,joshua king



  USL Dunkerque  —  SF sin salario:


,player,minutesPlayed
0,Naatan Skyttä,9


  CG plantilla completa:


,player,player_norm


In [20]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 0


In [21]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 475/542 (87.6%)


In [22]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [23]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_france_2425.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_france_2425.csv
   Jugadores totales:  542
   Con salario:        475
   Sin salario (NaN):  67
   Columnas:           121
